In [1]:
import boto3
import sagemaker
import os

# --- Configuration ---
# Replace these with your specific bucket paths if different
RAW_DATA_S3 = "s3://echodata25/results/mimic-iv-echo-raw/"
PROCESSED_DATA_S3 = "s3://echodata25/results/mimic-iv-echo-processed/"

# Docker Image Name
ECR_REPO_NAME = "mimic-echo-processor"

# Get Current Session Info
sess = sagemaker.Session()
role = sagemaker.get_execution_role()
region = boto3.Session().region_name
account_id = boto3.client('sts').get_caller_identity().get('Account')

# Full Image URI
image_uri = f"{account_id}.dkr.ecr.{region}.amazonaws.com/{ECR_REPO_NAME}:latest"

print(f"Role: {role}")
print(f"Region: {region}")
print(f"Target Image URI: {image_uri}")

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml
Role: arn:aws:iam::495467399120:role/service-role/AmazonSageMaker-ExecutionRole-20250409T120880
Region: us-west-2
Target Image URI: 495467399120.dkr.ecr.us-west-2.amazonaws.com/mimic-echo-processor:latest


In [2]:
%%writefile Dockerfile
FROM python:3.10-slim

# 1. Install System Dependencies
# ffmpeg: for video encoding
# libsm6, libxext6: required by OpenCV
# libgdcm-tools: required for compressed DICOMs
RUN apt-get update && apt-get install -y \
    ffmpeg \
    libsm6 \
    libxext6 \
    build-essential \
    && rm -rf /var/lib/apt/lists/*

# 2. Install Python Packages
RUN pip install --no-cache-dir \
    pydicom \
    numpy \
    opencv-python-headless \
    tqdm \
    boto3

# 3. Entrypoint
ENTRYPOINT ["python3"]

Writing Dockerfile


In [3]:
%%writefile process.py
import os
import glob
import pydicom
import cv2
import numpy as np
from tqdm import tqdm
import sys

# --- Config ---
# SageMaker automatically mounts S3 inputs to this local path
INPUT_DIR = "/opt/ml/processing/input"
# SageMaker automatically uploads files from this local path to S3
OUTPUT_DIR = "/opt/ml/processing/output"

TARGET_RES = (112, 112)

def process_and_save(dcm_path):
    """Reads DICOM, resizes to 112x112, saves as MP4."""
    try:
        # 1. Read DICOM
        dcm = pydicom.dcmread(dcm_path)
        pixel_array = dcm.pixel_array
        
        # 2. Normalize Shape to (Frames, H, W, 3)
        # Handle Grayscale (H, W) -> (1, H, W, 3)
        if len(pixel_array.shape) == 2:
            pixel_array = np.stack([pixel_array]*3, axis=-1)
            pixel_array = np.expand_dims(pixel_array, axis=0)
            
        # Handle (Frames, H, W) or (H, W, C)
        elif len(pixel_array.shape) == 3:
            # If Multi-frame grayscale (Frames, H, W) -> Convert to RGB
            if dcm.NumberOfFrames > 1 and pixel_array.shape[0] == dcm.NumberOfFrames:
                pixel_array = np.stack([pixel_array]*3, axis=-1)
            # If Single Frame RGB (H, W, 3) -> Add frame dim
            elif pixel_array.shape[2] == 3:
                 pixel_array = np.expand_dims(pixel_array, axis=0)

        frames, h, w, c = pixel_array.shape
        
        # 3. Define Output Path
        # e.g. /opt/ml/processing/output/study_123.mp4
        filename = os.path.splitext(os.path.basename(dcm_path))[0] + ".mp4"
        output_path = os.path.join(OUTPUT_DIR, filename)

        # 4. Write MP4
        # 'mp4v' is fast and compatible. Use 'avc1' if you need strict H.264
        out = cv2.VideoWriter(output_path, cv2.VideoWriter_fourcc(*'mp4v'), 30.0, TARGET_RES)
        
        for i in range(frames):
            frame = pixel_array[i]
            # Resize
            frame_resized = cv2.resize(frame, TARGET_RES, interpolation=cv2.INTER_AREA)
            out.write(frame_resized.astype('uint8'))
        
        out.release()
        return True
        
    except Exception as e:
        # Log error but don't crash the job
        print(f"[ERROR] Failed {dcm_path}: {e}")
        return False

def main():
    # Find all .dcm files in the input directory (recursive)
    files = glob.glob(f"{INPUT_DIR}/**/*.dcm", recursive=True)
    print(f"Found {len(files)} DICOMs to process in this shard.")
    
    success_count = 0
    
    # Process files
    for dcm_file in tqdm(files, desc="Converting"):
        if process_and_save(dcm_file):
            success_count += 1

    print(f"Shard Complete. Processed {success_count}/{len(files)} files.")

if __name__ == "__main__":
    main()

Writing process.py


In [4]:
from sagemaker.processing import ScriptProcessor, ProcessingInput, ProcessingOutput

# --- Job Configuration ---
INSTANCE_COUNT = 10           # Number of parallel nodes (1/10th of data each)
INSTANCE_TYPE = "ml.c5.4xlarge" # Compute optimized instance
TIMEOUT_SEC = 86400           # 24 hours max runtime

# Initialize the Processor
processor = ScriptProcessor(
    command=['python3'],
    image_uri=image_uri,
    role=role,
    instance_count=INSTANCE_COUNT,
    instance_type=INSTANCE_TYPE,
    max_runtime_in_seconds=TIMEOUT_SEC
)

print(f"Launching Processing Job with {INSTANCE_COUNT} instances of {INSTANCE_TYPE}...")

# Run the Job
processor.run(
    code='process.py', # Uploads your local process.py to the container
    
    inputs=[
        ProcessingInput(
            source=RAW_DATA_S3,
            destination='/opt/ml/processing/input',
            # CRITICAL: This ensures each instance gets a unique slice of files
            s3_data_distribution_type='ShardedByS3Key' 
        )
    ],
    
    outputs=[
        ProcessingOutput(
            source='/opt/ml/processing/output',
            destination=PROCESSED_DATA_S3,
            # CRITICAL: Uploads MP4s immediately upon creation
            s3_upload_mode='Continuous' 
        )
    ],
    
    # wait=False allows you to continue using the notebook, 
    # but wait=True streams logs here so you can see the progress bar.
    wait=True
)

Launching Processing Job with 10 instances of ml.c5.4xlarge...


INFO:sagemaker:Creating processing-job with name mimic-echo-processor-2026-01-13-23-23-09-050
ERROR:sagemaker:Please check the troubleshooting guide for common errors: https://docs.aws.amazon.com/sagemaker/latest/dg/sagemaker-python-sdk-troubleshooting.html#sagemaker-python-sdk-troubleshooting-create-processing-job


╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:21                                                                                   │
│                                                                                                  │
│   18 print(f"Launching Processing Job with {INSTANCE_COUNT} instances of {INSTANCE_TYPE}...")    │
│   19                                                                                             │
│   20 # Run the Job                                                                               │
│ ❱ 21 processor.run(                                                                              │
│   22 │   code='process.py', # Uploads your local process.py to the container                     │
│   23 │                                                                                           │
│   24 │   inputs=[                                                                                │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/sagemaker/workflow/pipeline_context.py:346 in wrapper    │
│                                                                                                  │
│   343 │   │   │                                                                                  │
│   344 │   │   │   return _StepArguments(retrieve_caller_name(self_instance), run_func, *args,    │
│   345 │   │                                                                                      │
│ ❱ 346 │   │   return run_func(*args, **kwargs)                                                   │
│   347 │                                                                                          │
│   348 │   return wrapper                                                                         │
│   349                                                                                            │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/sagemaker/processing.py:681 in run                       │
│                                                                                                  │
│    678 │   │   )                                                                                 │
│    679 │   │                                                                                     │
│    680 │   │   experiment_config = check_and_get_run_experiment_config(experiment_config)        │
│ ❱  681 │   │   self.latest_job = ProcessingJob.start_new(                                        │
│    682 │   │   │   processor=self,                                                               │
│    683 │   │   │   inputs=normalized_inputs,                                                     │
│    684 │   │   │   outputs=normalized_outputs,                                                   │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/sagemaker/processing.py:917 in start_new                 │
│                                                                                                  │
│    914 │   │   logger.debug("Outputs: %s", process_args["output_config"]["Outputs"])             │
│    915 │   │                                                                                     │
│    916 │   │   # Call sagemaker_session.process using the arguments dictionary.                  │
│ ❱  917 │   │   processor.sagemaker_session.process(**process_args)                               │
│    918 │   │                                                                                     │
│    919 │   │   return cls(                                                                       │
│    920 │   │   │   processor.sagemaker_session,            